# Bölüm 12 · Regresyon Analizi

[Atölye](https://ibrahimguney.github.io/ileri-istatistik-uygulamalari/surumler/22-bolum/bolumler/b12/) · [Veri ve lisans](https://ibrahimguney.github.io/ileri-istatistik-uygulamalari/surumler/22-bolum/bolumler/b12/VERI.md) · [Görevler](https://ibrahimguney.github.io/ileri-istatistik-uygulamalari/surumler/22-bolum/bolumler/b12/GOREVLER.md) · [Raporlama](https://ibrahimguney.github.io/ileri-istatistik-uygulamalari/surumler/22-bolum/bolumler/b12/RAPORLAMA.md) · [Çözümler](https://ibrahimguney.github.io/ileri-istatistik-uygulamalari/surumler/22-bolum/bolumler/b12/COZUMLER.md)

**Başlangıç:** Google hesabınızla açın, **Dosya → Drive'a bir kopyasını kaydet** seçin. CPU yeterlidir. Hücreleri sırayla veya **Çalışma zamanı → Tümünü çalıştır** ile yürütün. İlk kurulum internet ve birkaç dakika gerektirebilir. Özellikle bootstrap ve yapısal modeller daha uzun sürebilir.

Bu defter, aynı GitHub deposundaki ana Python analizini ve bölümün ileri eklerini çalıştırır. Dosya yüklemeniz veya önce başka bölümü elle yürütmeniz gerekmez. Gerçek veri, öğretim alt kümesi ve simülasyon ayrımı veri rehberinde açıklanır; bunları aynı örneklem türü saymayın. R/SPSS/Amos burada çalıştırılmaz.

## Üç çekirdek kazanım ve ileri rota

1. Regresyon katsayısını bağlam ve birimle yorumlar.
2. Artık, etkili gözlem ve genelleme sınırlarını inceler.
3. Ortalama güven aralığı ile bireysel tahmini ayırır.

**İleri rota:** HC3, LOOCV ve split conformal ayrı ileri rotalardır; eğitim/kalibrasyon/kontrol rolleri karıştırılmaz.

**Ç1 · Çıkış sorusu:** Ortalama yanıt için dar güven aralığı yeni öğrencinin notunun aynı dar aralıkta olacağını garanti eder mi?

Yanıtınızı çalışma sonunda öğrenci karar günlüğüne yazın. [Rubrik ve teslim rehberi](https://ibrahimguney.github.io/ileri-istatistik-uygulamalari/surumler/22-bolum/bolumler/b12/RUBRIK.md).

## 1 · Kaynakları hazırla

Sabit bir GitHub sürümündeki öğrenci paketi SHA256 ile doğrulanır. Her çalıştırma ayrı bir geçici çalışma kopyası açar; önceki sonuçlarınız silinmez. Kod ve veri, bu doğrulanan sürümden gelir. Oturum kapanmadan sonuçları indirin.

In [ ]:
from pathlib import Path
import sys, os, json, hashlib, urllib.request, zipfile, io, tempfile, subprocess, importlib.metadata
DATA_COMMIT = '23acf4be23823aa139523c8f4db34cff248c285d'
ARCHIVE_SHA256 = 'ce6440f5ee3b4dde9709e1fb250e7b7d8ac86be101b6ababacdab240443560f6'
ARCHIVE_URL = f"https://raw.githubusercontent.com/ibrahimguney/ileri-istatistik-uygulamalari/{DATA_COMMIT}/surumler/22-bolum/ogrenci-paketi-22-bolum.zip"
cache = Path.cwd()/"kitap-colab-kaynak.zip"
if cache.exists():
    raw = cache.read_bytes()
else:
    try:
        with urllib.request.urlopen(ARCHIVE_URL, timeout=60) as response: raw=response.read()
    except Exception as exc:
        raise RuntimeError("Kaynak indirilemedi. İnterneti kontrol edip hücreyi yeniden çalıştırın: "+ARCHIVE_URL) from exc
if hashlib.sha256(raw).hexdigest()!=ARCHIVE_SHA256:
    raise ValueError("Öğrenci paketi SHA256 denetimi başarısız. Kontrolü kaldırmayın; yerel kitap-colab-kaynak.zip dosyasını doğrulayın.")
cache.write_bytes(raw)
work = Path(tempfile.mkdtemp(prefix="b12-atolye-",dir=Path.cwd()))
with zipfile.ZipFile(io.BytesIO(raw)) as z:
    assert len(z.infolist())<10000 and sum(i.file_size for i in z.infolist())<134217728
    for member in z.infolist():
        target=(work/member.filename).resolve()
        if work.resolve() not in target.parents or ((member.external_attr>>16)&0o170000)==0o120000:
            raise ValueError("Güvenli olmayan arşiv yolu")
    z.extractall(work)
root = work/"ileri-istatistik-22-bolum"
assert (root/"BOLUM_ESLESMESI.json").is_file()
application=root/'uygulamalar/b12-regresyon'
modules=['uygulamalar/b12-regresyon/ayri-kalibrasyon']
expected_main=json.loads((root/"bolumler/b12/beklenen.json").read_text())
expected_extra={m:json.loads((root/m/"sonuclar/ozet.json").read_text()) for m in modules}
print("Kaynak doğrulandı; Bölüm 12 çalışma kopyası hazır.")

## 2 · Analiz ortamını hazırla

Sınanmış paket sürümleri kullanılır. Mevcut çekirdek bu sürümlerdeyse doğrudan kullanılır. Farklıysa analizler ayrı bir Python ortamında kurulur; Colab çekirdeğindeki paketler topluca değiştirilmez. Kurulum başarısız olursa hücredeki hatayı inceleyin. Python 3.12 veya üzeri çalışma zamanı önerilir. GPU gerekmez.

In [ ]:
FORCE_ISOLATED=globals().get("FORCE_ISOLATED",False)
PINS={'numpy': '2.5.4', 'scipy': '1.18.1', 'pandas': '3.1.0', 'matplotlib': '3.11.2', 'statsmodels': '0.15.0', 'PyMuPDF': '1.26.6'}
def package_versions(executable):
    check="import importlib.metadata,json; names="+repr(list(PINS))+";print(json.dumps({n:importlib.metadata.version(n) for n in names}))"
    attempt=subprocess.run([str(executable),"-c",check],capture_output=True,text=True)
    return json.loads(attempt.stdout) if attempt.returncode==0 else {}
python=Path(sys.executable)
if FORCE_ISOLATED or package_versions(python)!=PINS:
    environment=Path.cwd()/".kitap-colab-python"
    if not (environment/"bin/python").exists():
        subprocess.run([sys.executable,"-m","venv",str(environment)],check=True)
    python=environment/"bin/python"
    if package_versions(python)!=PINS:
        subprocess.run([str(python),"-m","pip","install",*[name+"=="+version for name,version in PINS.items()]],check=True)
assert package_versions(python)==PINS,"Paket sürümleri sınanmış ortamla eşleşmiyor"
versions=package_versions(python)
run_env={**os.environ,"MPLBACKEND":"Agg","MPLCONFIGDIR":str(work/"mpl"),"OPENBLAS_NUM_THREADS":"1","OMP_NUM_THREADS":"1","PYTHONDONTWRITEBYTECODE":"1"}
(work/"logs").mkdir(exist_ok=True)
def run_script(relative):
    print("Çalışıyor:",relative,flush=True)
    result=subprocess.run([str(python),str(root/relative)],cwd=root,env=run_env,capture_output=True,text=True,timeout=900)
    (work/"logs"/(relative.replace("/","__")+".log")).write_text(result.stdout+"\n"+result.stderr)
    if result.returncode: raise RuntimeError(result.stdout[-4000:]+"\n"+result.stderr[-4000:])
    print("Tamamlandı.")
print("Analiz ortamı hazır:",versions)

## Ana analiz

Kaynak ve yöntem için yukarıdaki veri rehberini okuyun. Ana analiz ve sonradan eklenen duyarlılık uygulamaları ayrı çalıştırılır. Sonuçlar önceden gösterilen tabloların kopyası olarak değil, kod yeniden yürütülerek üretilir.

In [ ]:
before_figures=set((root/"assets_figures").glob("*.pdf"))
run_script("bolumler/b12/analiz.py")
main_result=json.loads((application/"sonuclar/ozet.json").read_text())
from IPython.display import display, JSON, HTML, Image
print("Ana sonuçlar — ayrıntıları açmak için okları kullanın:")
display(JSON(main_result,expanded=False))

## İleri uygulamalar ve duyarlılık

Her ekin açıklaması atölye sayfasındadır. Bu sonuçları ana analizle veya birbirinden farklı tahmin hedefleriyle karıştırmayın. Hazır kontrol betikleri varsa önce onlar, ardından ek analiz çalışır.

In [ ]:
extra_results={}
for module in modules:
    if (root/module/"dogrula.py").is_file(): run_script(module+"/dogrula.py")
    run_script(module+"/analiz.py")
    extra_results[module]=json.loads((root/module/"sonuclar/ozet.json").read_text())
    print(module)
    display(JSON(extra_results[module],expanded=False))

## Sonuçları referanslarla karşılaştır

Kitabın sayısal referansları, önceki sürümden değişmeden alınır. Sürüm/zaman gibi hesap dışı alanlar mevcut karşılaştırma politikasına göre dışlanır. Hata oluşursa veri, sürüm ve günlük dosyalarını kontrol edin; referansları sonuca uydurmayın. Bu kontrol bilimsel varsayımları veya SPSS yürütmesini doğrulamaz.

In [ ]:
sys.path.insert(0,str(root/"tools"))
from dogrula_22 import compare
checks={"main":compare(expected_main,main_result)}
for module in modules:
    checks[module]=compare(expected_extra[module],json.loads((root/module/"sonuclar/ozet.json").read_text()))
assert all(not differences for differences in checks.values()),checks
print("Ana analiz ve",len(modules),"ileri ek için sayısal referanslar eşleşti.")

## Grafikler ve özet tablolar

Grafikler bu çalıştırmada üretilir. Ön koşul olarak çalıştırılan Bölüm 18'in şekilleri burada tekrar gösterilmez. CSV tablolarının ilk 12 satırı görüntülenir; tam dosyalar çıktı paketindedir. Şekil sayısı bölüme göre değişir.

In [ ]:
import csv,html
new_figures=sorted(set((root/"assets_figures").glob("*.pdf"))-before_figures)
for figure in new_figures:
    png=figure.with_suffix(".png")
    renderer="import fitz,sys;d=fitz.open(sys.argv[1]);d[0].get_pixmap(matrix=fitz.Matrix(1.4,1.4)).save(sys.argv[2])"
    subprocess.run([str(python),"-c",renderer,str(figure),str(png)],check=True)
    print(figure.name)
    display(Image(filename=str(png)))
for table in sorted((application/"sonuclar").glob("*.csv"))[:3]:
    with table.open(encoding="utf-8-sig",newline="") as f: rows=list(csv.reader(f))
    content="<h4>"+html.escape(table.name)+"</h4><table>"
    for i,row in enumerate(rows[:13]):
        tag="th" if i==0 else "td"
        content+="<tr>"+"".join("<"+tag+">"+html.escape(value)+"</"+tag+">" for value in row)+"</tr>"
    display(HTML(content+"</table>"))

## Sıra sizde

# Bölüm 12 — görevler

Güncel Revizyon 15 görevleri; önce kendiniz çözün.

## Hesaplamadan Önce: Tahmin Et ve Gerekçelendir

1.  **\[Yöntem seç\]** Açıklanan ve açıklayıcı değişkeni belirleyin. Neden iki grup ortalaması karşılaştırması yerine basit regresyon düşünürsünüz? Bu tahmin hangi zamanda kullanılabilir?

2.  **\[Yorumla\]** Eğimin işaretini tahmin edin. Birinci dönem notu daha yüksek olan herkesin yıl sonu notunun da daha yüksek olması gerekir mi? Tablodaki kayıtlarla tartışın.

3.  **\[Eleştir\]** Önceki ve sonraki notların ilişkisini etkileyebilecek iki etmen önerin. Bu ilişki, birinci dönem notunu idari olarak yükseltmenin yıl sonu başarısını artıracağını gösterir mi? Neden?

Yanıtlarınızı yazdıktan sonra ilerleyin. Etkinliklerin gerekçeli yanıtları Ek F’dedir (s. kitaptaki ilgili sayfa).

# Rehberli ve Bağımsız Etkinlikler

## Rehberli Uygulama: Şimdi Siz Tamamlayın

Birinci dönem notu 12 olan bir öğrenci düşünün. Cevap anahtarını açmadan aşağıdaki adımları tamamlayın.

1.  **\[Hesapla ve yorumla\]** $\hat{y}_0=5.7412+0.5820(12)=\ldots$ bulunur. Kaynak sırası 3 olan öğrencinin yıl sonu notu 12’dir; artık $12-\hat{y}_0=\ldots$ olur. Nokta tahmini bu öğrencinin kesin notu mudur?

2.  **\[İleri: hesapla\]** $h_0=1/60+(12-12.1)^2/385.4=0.0166926$ veriliyor. $s_e=1.3368$ ve $t^*=2.0017$ ile ortalama yanıtın ve yeni bireyin klasik %95 aralıklarını hesaplayın. Bu aralıkların geçerliliğini hangi tanısal bulgular nedeniyle temkinli yorumlamalısınız?

3.  **\[İleri: yorumla\]** $x_0=\bar{x}$ ve $x_0=17$ için aralık genişliğini karşılaştırın. Ortalama tahmini nerede en hassastır? Not ölçeğinde mümkün olan 20, bu alt kümede gözlenmiş midir? Formüle yazılabilmesi güvenilir tahmin için yeterli midir?

## Bağımsız Uygulama: Sonraki 20 Gerçek Kayıt

`aktarim.csv`, aynı resmi kaynağın 61–80. sıralarındaki 20 gerçek kaydı içerir. Buradaki “bağımsız”, alıştırmayı yardım almadan çözmeyi belirtir; istatistiksel bağımsızlık veya dış doğrulama tasarımı iddiası değildir. Bu kayıtlar da GP okuluna aittir. Ana veriye eklemeyin ve iki ayrı örnekleme yapılmış gibi yorumlamayın.

1.  **\[Analiz et\]** Saçılım grafiğini çizip $G3\sim G1$ sabit terimli modelini kurun. $n$, $b_0$, $b_1$ ve $R^2$ değerlerini bulun; kodunuzu veya hesaplama yolunuzu ekleyin.

2.  **\[Hesapla\]** $G1=12$ için tahmini ve kaynak sırası 80 olan kaydın ($G3=11$) artığını hesaplayın. Yalnızca açıklayıcı notu 100’lük ölçeğe, $M=5G1$ biçiminde dönüştürün. Yanıt G3 hâlâ 20’lik ölçekte kalsın. Eğim, sabit, tahminler ve $R^2$ nasıl değişir?

3.  **\[Raporla\]** En fazla 100 sözcüklük sonuç yazın. İlk 60 ve sonraki 20 kaydın eğimleri farklıysa bu, iki öğrenci topluluğu arasında gerçek fark kanıtı mıdır? Seçim kuralını ve belirsizliği dikkate alarak açıklayın.

## Hatalı Raporu Düzeltme Atölyesi

**H1 \[Eleştir ve yeniden yaz\].** Aşağıdaki metin **kasıtlı olarak hatalıdır**. En az beş sorunu belirleyip gerekçeleriyle düzeltin.

> “Birinci dönem notunu bir puan artırmak her öğrencinin yıl sonu notunu kesinlikle 0.582 artırır. $R^2=0.557$, öğrencilerin yüzde 55.7’sinin başarılı olacağını gösterir. $p<.001$ olduğundan bütün varsayımlar sağlanmıştır. G1 notu 12 olan yeni öğrencinin yıl sonu notu yüzde 95 güvenle 12.38–13.07 arasındadır. Aynı model G1=20 için de aynı güvenilirliktedir. Bu bulgular Türkiye’deki bütün öğrenciler için geçerlidir. İlk kayıttaki sıfır notu hatadır; onu silerek daha doğru sonucu buluruz.”

Bir arkadaşınızın düzeltmesinde nedensellik dili, aralık türü, etkili gözlem kararı ve örnekleme sınırlarını kontrol edin. Kaynağa atıf verilmesi, bu hatalı yorumları geçerli kılmaz.

## T1–T3: Eğitim, Kalibrasyon ve Tahmin Denetimi

1.  Eğitim 1–40, kalibrasyon 41–60, kontrol 61–80 rollerini açıklayın. Kalibrasyondan sonra 60 kayıtta yeniden uyum veya kontrol yanıtlarına göre eşik seçimi neden aynı geçerli split conformal hesabı değildir?

2.  $m=20$, $\alpha=.05$ için k ve q’yu; $m=18$ için k’yi bulun. Sıradan enterpolasyonlu yüzdelik neden kullanılmaz? G1=12 için aralığı model ve q’dan hesaplayın.

3.  Son 20 kayıtta 18/20 kapsama ne söyler, ne söylemez? Marjinal, x’e koşullu ve eşzamanlı kapsama ayrımını yapın; HC3 eğim aralığı neden bunların yerine geçmez?

## Mini Proje: Yeniden Üretilebilir Analiz Raporu

**P1 \[Analiz et ve raporla\].** Ana senaryodaki veriyle 2–3 sayfalık bir analiz raporu hazırlayın. Raporunuza çalıştırılabilir kod veya SPSS sözdizimi ekleyin; yalnızca ekran görüntüsü teslim etmeyin.

1.  Araştırma sorusunu, değişkenleri, birimleri, UCI kaynağını ve ilk 60 kayıt seçim kuralını açıklayın. Atıf, bağımsızlık ve genelleme sınırlarını belirtin.

2.  Saçılım ve artık grafiklerini üretin. Modeli kurup katsayıları, eğim güven aralığını, $R^2$’yi ve artık standart sapmasını raporlayın. Gözlem sayısını ve yuvarlama tercihinizi yazın.

3.  G1=12 için klasik ortalama güven aralığını ve bireysel tahmin aralığını hesaplayın; öğretim elemanına hangi sorunun hangi aralıkla yanıtlandığını açıklayın.

4.  Etki duyarlılığı için her seferinde bir gözlemi dışarıda bırakarak 60 model kurun. Eğimlerin aralığını gösterin. Bir gözlemi silmeye karar vermek ile duyarlılığı araştırmak arasındaki farkı tartışın; gözlemleri yalnızca $p$-değerini küçültmek için çıkarmayın.

5.  Sonuç, sınırlılıklar ve yeniden üretim yönergeleri yazın. Analiz dosyaları, veri sözlüğü ve kullanılan yazılım sürümlerini teslim paketine ekleyin.

### İleri Rota: Açıklama ile Yeni Gözlem Tahminini Ayırma

Aynı veride modelin eğitim hatasını ve *birini dışarıda bırakmalı çapraz doğrulama* (LOOCV) hatasını karşılaştırın. Her döngüde bir kaydı ayırın, modeli kalan 59 kayıtla **yeniden kurun** ve ayrılan kaydın puanını tahmin edin. Sonra $$RMSE_{\text{eğitim}}=\sqrt{\frac{1}{n}\sum e_i^2},\qquad
RMSE_{\mathrm{LOOCV}}=\sqrt{\frac{1}{n}\sum(y_i-\hat{y}_{i,(-i)})^2}$$ hesaplayın. $\hat{y}_{i,(-i)}$, $i$. kayıt kullanılmadan kurulan modelin o kayıt için tahminidir. Eğitim RMSE’sindeki paydanın $n$, artık standart sapmasındaki paydanın $n-2$ olduğuna dikkat edin. Eğitimde açıklanan varyans ile ayrılmış bir gözleme ilişkin tahmin performansının aynı soru olmadığını tartışın. Sıralı tek okul alt kümesindeki çapraz doğrulama, yeni okullardaki veya daha sonraki yıllardaki başarıyı garanti etmez. İleri rotada klasik ve HC3 eğim standart hatalarını da karşılaştırın; HC3 hesaplamasının bireysel tahmin aralığını kendiliğinden düzeltmediğini açıklayın.

| Puan | Tam puan için beklenen kanıt                                                             |
|:-----|:-----------------------------------------------------------------------------------------|
| 15   | Soru, değişken rolleri, birimler ve gerçek veri kaynağı/seçim kuralı açık.               |
| 20   | Model, katsayılar ve temel hesaplamalar doğru ve izlenebilir.                            |
| 20   | Grafikler yorumlanmış; duyarlılık ve varsayım sınırları tartışılmış.                     |
| 15   | İki aralık doğru hesaplanmış ve doğru sorularla eşleştirilmiş.                           |
| 20   | Rapor bağlama uygun; nedensellik, başarı olasılığı ve genelleme hatalarından kaçınılmış. |
| 10   | Veri, kod, sürüm ve çalıştırma yönergeleriyle sonuç yeniden üretilebilir.                |

Mini proje değerlendirme ölçütleri: toplam 100 puan

Her ölçütte tam ve gerekçeli kanıt tam puan; kısmen doğru fakat eksik gerekçe yarım puan; yanlış veya sunulmamış kanıt sıfır puan olarak değerlendirilir. İleri rota uzantısı temel rota notuna ek zorunluluk getirmez; ileri rotada tanısal değerlendirme ölçütünün içinde ele alınır.

# Alıştırmalar

Bu alıştırmalar, basit doğrusal regresyonun katsayı yorumu, model uyumu, varsayım denetimi ve raporlama becerilerini pekiştirmek için üç düzeyde düzenlenmiştir. Yanıtlar ve kısa çözüm ipuçları Ek F’dedir.

## Kolay Düzey

1.  Basit regresyonda eğim katsayısı neyi ifade eder?

2.  $R^2$ değeri neyi gösterir?

3.  Basit regresyonda genel model anlamlılığı hangi testle değerlendirilebilir?

4.  Artık değeri nasıl tanımlanır?

5.  Basit regresyonda sabit terim ($b_0$) neyi ifade eder ve yorumu hangi koşulda anlamlıdır?

## Orta Düzey

1.  Basit regresyonda $R^2$ ile korelasyon katsayısı arasında nasıl bir ilişki vardır?

2.  Bir araştırmada çalışma süresi ($X$) ile sınav puanı ($Y$) arasındaki ilişki için şu regresyon denklemi elde edilmiştir: $\hat{Y}=40+3X$. Eğim ve sabit terimi açıklayınız.

3.  Bir regresyon analizinde $R^2=0.58$ bulunmuştur. Bu değeri açıklayınız.

4.  Bir regresyon modelinde artık grafiği huni biçiminde görünüyorsa hangi varsayım ihlalinden söz edilebileceğini belirtiniz.

5.  Bir veri setinde $\sum(x_i-\bar{x})(y_i-\bar{y})=30$, $\sum(x_i-\bar{x})^2=10$, $\bar{x}=4$ ve $\bar{y}=10$ verilmiştir. Eğim ve sabit terimi hesaplayınız.

## İleri Düzey

1.  $X$ değişkeninin 2–7 aralığında gözlendiği bir modelle $X=20$ için tahmin üretmek neden risklidir?

2.  $R^2$ oldukça yüksek olduğu hâlde artık grafiği belirgin eğrisel bir desen gösteriyorsa ne düşünülmelidir?

3.  Tek bir yüksek kaldıraçlı gözlemin eğim katsayısını ciddi biçimde değiştirdiği görülmüştür. Araştırmacı sonucu raporlamadan önce hangi adımları izlemelidir?

4.  Aynı $X$ düzeyi için ortalama yanıtın güven aralığı ile tek bir yeni gözlemin tahmin aralığından hangisi daha geniştir? Nedenini açıklayınız.

5.  Güçlü bir basit regresyon raporunda bulunması gereken en az beş unsuru yazınız.


Önce kendi yorumunuzu yazın; sonra [çözümlerle](https://ibrahimguney.github.io/ileri-istatistik-uygulamalari/surumler/22-bolum/bolumler/b12/COZUMLER.md) karşılaştırın. Sonuç, belirsizlik, varsayım ve sınırlılıkları [raporlama rehberine](https://ibrahimguney.github.io/ileri-istatistik-uygulamalari/surumler/22-bolum/bolumler/b12/RAPORLAMA.md) göre ayırın.

In [ ]:
yanitlarim={"bulgular":"", "belirsizlik":"", "varsayimlar":"", "sinirliliklar":""}
# Metinleri doldurun; sayısal kontrolü geçmek bu yorumları tamamlamaz.

## Çıktıları indir

Son hücre sonuç JSON/CSV'lerini, grafiklerin PDF/PNG dosyalarını, günlükleri ve kendi yanıtlarınızı paketler. Colab'ın solundaki Dosyalar panelini yenileyin ve aşağıda yazan ZIP'i indirin. Geçici çalışma zamanı kapanmadan bunu yapın; kod Drive'a otomatik erişmez.

## Öğrenci kararı ve kısa savunma

**Karar görevi:** Ortalama yanıt veya yeni birey tahmini hedefini seçin; G1=12 için uygun aralık türünü ve genişliğini karşılaştırın.

Yalnız bir kararı değiştirin. Ana sonucu koruyun; gerekirse yeni bir hesap hücresi ekleyip denemeyi ayrı kaydedin. Sayısal referansları yeni sonuca uydurmayın. Önce/sonra kanıtını sonuç dosyası, tablo veya açık bir hesapla ilişkilendirin. Yapılmamış hesabı yapılmış gibi yazmayın.

**3–5 dakikalık savunma:** Hangi hedefi seçtiniz ve neden? Tam olarak ne değişti? Hangi belirsizlik ve kapsam sınırı kaldı? Karşılaştırma aynı hipotezi mi sınadı?

Aşağıdaki alanları kendi çalışmanızla doldurun. **Tümünü çalıştır**, bu alanları sizin yerinize tamamlamaz. Sayısal kontrollerin geçmesi öğrenci kararının tamamlandığı anlamına gelmez. Alanların dolu olması da kanıtın doğruluğunu otomatik onaylamaz.

In [ ]:
karar_gunlugu = {
    "bolum": 12,
    "gorev": 'Ortalama yanıt veya yeni birey tahmini hedefini seçin; G1=12 için uygun aralık türünü ve genişliğini karşılaştırın.',
    "kararim": "",
    "gerekcem": "",
    "degistirdigim_tek_sey": "",
    "onceki_kanit": "",
    "sonraki_kanit": "",
    "belirsizlik_ve_sinir": "",
    "kisa_savunma": "",
    "cikis_yanitim": ""
}
zorunlu_alanlar = ["kararim","gerekcem","degistirdigim_tek_sey","onceki_kanit","sonraki_kanit","belirsizlik_ve_sinir","kisa_savunma","cikis_yanitim"]
karar_gunlugu["alanlar_dolu"] = all(isinstance(karar_gunlugu[k],str) and bool(karar_gunlugu[k].strip()) for k in zorunlu_alanlar)
karar_gunlugu["durum"] = "Öğretim elemanı incelemesi gerekir" if karar_gunlugu["alanlar_dolu"] else "Öğrenci kararı henüz tamamlanmadı"
print(karar_gunlugu["durum"])
# Bu hücreyi doldurduktan sonra çıktı ZIP'i hücresini yeniden çalıştırın.

In [ ]:
manifest={"chapter":12,"source_commit":DATA_COMMIT,"source_archive_sha256":ARCHIVE_SHA256,"versions":versions,"checks":checks,"answers":yanitlarim,"scope":"Python ana analiz ve ileri ekler; SPSS/R/Amos çalıştırılmadı"}
# OGRETIM_EXPORT_BEGIN
manifest["ogrenci_karari"] = karar_gunlugu
(work/"ogrenci-karari.json").write_text(json.dumps(karar_gunlugu,ensure_ascii=False,indent=2),encoding="utf-8")
# OGRETIM_EXPORT_END
(work/"calisma-kaydi.json").write_text(json.dumps(manifest,ensure_ascii=False,indent=2))
archive=Path.cwd()/"b12-colab-ciktilar.zip"
with zipfile.ZipFile(archive,"w",zipfile.ZIP_DEFLATED) as z:
    z.write(work/"ogrenci-karari.json","ogrenci-karari.json")
    z.write(work/"calisma-kaydi.json","calisma-kaydi.json")
    for module in ['uygulamalar/b12-regresyon',*modules]:
        for f in sorted((root/module/"sonuclar").rglob("*")):
            if f.is_file() and f.suffix in [".json",".csv"]:z.write(f,str(f.relative_to(root)))
    for figure in new_figures:
        for f in [figure,figure.with_suffix(".png")]: z.write(f,"grafikler/"+f.name)
    for f in (work/"logs").glob("*.log"):z.write(f,"logs/"+f.name)
print("İndirmeye hazır:",archive.name)